# Chapter 6: What a Layer Actually Does

## The versor decomposition $T = RP$, rotor field extraction, grade-0 vs grade-2

In Chapter 5 we learned that rotations are rotors — geometric objects with a plane and an angle. But a real transformer layer does more than rotate: it also **stretches** representations, amplifying some directions and suppressing others.

The full story of what a layer does is captured by the **versor decomposition**:

$$T^{(l)} = R^{(l)} \cdot P^{(l)}$$

This is the GA analog of the polar decomposition from linear algebra, reinterpreted through the lens of Clifford algebra:

| Component | GA name | Grade | What it does |
|:---|:---|:---|:---|
| $R^{(l)}$ | **Rotor** | Grade-2 (bivector generator) | Rotates representations in specific planes |
| $P^{(l)}$ | **Metric factor** | Grade-0 (eigenvalue spectrum) | Stretches/compresses along principal directions |

The **rotation** ($R$) preserves norms — it redirects information without changing magnitudes. The **metric factor** ($P$) changes magnitudes — it selectively amplifies or suppresses.

A key empirical finding: **modifying grade-0 (eigenvalues/stretching) accounts for ~48% of the output change, while modifying grade-2 (rotation) accounts for only ~9%**. The stretching is where the transformer exerts most of its control over representations. Rotation preserves norms and is therefore "cheaper" in terms of information processing; stretching is selective and lossy.

By the end of this chapter you will be able to:
- Extract the rotor field from a real transformer model
- Plot the rotation angle profile $\theta(l)$ across layers
- Compare the magnitudes of grade-0 (stretching) and grade-2 (rotation) at each layer
- Understand why the metric factor dominates the transformer's effect on representations

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga
from layer_time_ga.decomposition import extract_rotor_field
from layer_time_ga.algebra import grade_decomposition

np.set_printoptions(precision=4, suppress=True)
print("Imports OK")
print(f"NumPy {np.__version__}")

## The Versor Decomposition $T = RP$

Every layer transition matrix $T^{(l)}$ (the linear map from layer $l$ to layer $l+1$ in the whitened basis) admits a **polar decomposition**:

$$T^{(l)} = U^{(l)} \cdot P^{(l)}$$

where $U^{(l)}$ is orthogonal ($U^T U = I$, $\det U = +1$) and $P^{(l)}$ is symmetric positive-definite.

In GA language, we relabel this as the **versor decomposition**:

- $R^{(l)} = U^{(l)}$ is the **rotor** (grade-2 element). Its bivector generator $B^{(l)} = \log(R^{(l)})$ encodes the planes and angles of rotation.
- $P^{(l)}$ is the **metric factor** (grade-0 element). Its eigenvalues $\{\sigma_1, \ldots, \sigma_k\}$ are the **stretch spectrum** — how much each principal direction is amplified or suppressed.

The **condition number** $\kappa = \sigma_{\max} / \sigma_{\min}$ measures how *selective* the stretching is. $\kappa \approx 1$ means nearly isotropic (all directions treated equally). $\kappa \gg 1$ means the layer strongly favors some directions over others.

The **effective rank** $\text{erank} = \exp(H(\hat{\sigma}))$, where $H$ is the entropy of the normalized singular values, measures how many directions carry significant weight. Low effective rank means the stretching concentrates on a few directions.

Let us extract these quantities from a real transformer.

In [ ]:
# ── Load model and analyse a prompt ────────────────────────────
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")
print(f"Model: {model.name}")
print(f"  Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")

prompt1 = "The theory of general relativity describes gravity as the curvature of spacetime"
result1 = ltg_ga.analyse(prompt1, model=model)

# ── Print the versor decomposition table ───────────────────────
rf = result1.rotor_field

print(f"\nPrompt: \"{prompt1[:60]}...\"")
print(f"Tokens: {result1.n_tokens}  |  Layers: {result1.n_layers}  |  Whitened dim: {result1.k}")
print(f"\n{'Layer':>5s}  {'Angle (rad)':>11s}  {'Angle (°)':>9s}  {'κ':>8s}  {'erank':>7s}  {'rank':>5s}")
print("─" * 55)

for vd in rf.decompositions:
    angle_rad = vd.rotor.angle
    angle_deg = np.degrees(angle_rad)
    print(f"{vd.layer_index:5d}  {angle_rad:11.4f}  {angle_deg:9.1f}  "
          f"{vd.condition_number:8.2f}  {vd.effective_rank:7.1f}  {vd.rank:5d}")

## The Rotor Angle Profile

The rotor angle $\theta(l)$ at each layer tells us **how much rotation** the transformer applies at that transition. Plotting $\theta(l)$ across all layers reveals the geometric rhythm of the model:

- **Large angles** mean the layer significantly reorients representations — it is actively "steering" the hidden states into new directions.
- **Small angles** mean the rotation is gentle — the layer barely changes the orientation, perhaps fine-tuning or preserving structure from earlier layers.

The shape of $\theta(l)$ is remarkably consistent across prompts for a given model, though the details vary. It reflects the *architectural signature* of the model — how the model distributes its rotational work across depth.

In [ ]:
# ── Plot rotor angles across layers ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

layers = np.arange(len(result1.angles))
ax.plot(layers, result1.angles, color='#4477AA', linewidth=2,
        marker='o', markersize=3, label='Rotor angle θ(l)')
ax.fill_between(layers, result1.angles, alpha=0.15, color='#4477AA')

# Mark the peak
peak_idx = result1.angles.argmax()
ax.axvline(peak_idx, color='grey', linestyle='--', alpha=0.5,
           label=f'Peak at layer {peak_idx}')
ax.annotate(f'θ = {result1.angles[peak_idx]:.3f} rad',
            xy=(peak_idx, result1.angles[peak_idx]),
            xytext=(peak_idx + 2, result1.angles[peak_idx] * 1.05),
            fontsize=9, color='grey')

ax.set_xlabel('Layer transition', fontsize=11)
ax.set_ylabel('Rotation angle θ (rad)', fontsize=11)
ax.set_title(f'Rotor Angle Profile — {model.name}', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()

save_path = os.path.join(os.getcwd(), 'ch06_rotor_angles.png')
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()

## Grade-0 vs Grade-2: Which Dominates?

The versor decomposition splits each layer transition into two fundamentally different operations:

- **Grade-2 (rotation)**: encoded by the bivector $B^{(l)}$. We measure its magnitude by $\|B^{(l)}\|_F$ — the Frobenius norm of the skew-symmetric generator.
- **Grade-0 (stretching)**: encoded by the metric factor $P^{(l)}$. We measure its deviation from isotropy by $\|P^{(l)} - I\|_F$ — how far the stretch spectrum is from uniform.

When $\|P - I\|_F$ is large, the layer is *actively reshaping* magnitudes — some directions get amplified, others suppressed. When $\|B\|_F$ is large, the layer is *actively reorienting* directions — swinging representations into new planes.

Comparing these two quantities across layers reveals which type of geometric transformation dominates at each depth.

In [ ]:
# ── Compute grade-0 and grade-2 magnitudes per layer ───────────
rf = result1.rotor_field

grade0_norms = []  # ||P - I||_F  (metric deviation from identity)
grade2_norms = []  # ||B||_F      (bivector norm)
layer_indices = []

for vd in rf.decompositions:
    # Grade-2: bivector norm
    b_norm = vd.bivector.norm
    grade2_norms.append(b_norm)

    # Grade-0: metric deviation from identity
    P = vd.metric
    p_dev = np.linalg.norm(P - np.eye(P.shape[0]), 'fro')
    grade0_norms.append(p_dev)

    layer_indices.append(vd.layer_index)

grade0_norms = np.array(grade0_norms)
grade2_norms = np.array(grade2_norms)
layer_indices = np.array(layer_indices)

# ── Side-by-side bar chart ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grade-0 bars
ax1.bar(layer_indices, grade0_norms, color='#AA3377', alpha=0.8, width=0.8)
ax1.set_xlabel('Layer transition', fontsize=11)
ax1.set_ylabel('$\\|P^{(l)} - I\\|_F$', fontsize=11)
ax1.set_title('Grade-0: Metric Deviation (Stretching)', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')

# Grade-2 bars
ax2.bar(layer_indices, grade2_norms, color='#4477AA', alpha=0.8, width=0.8)
ax2.set_xlabel('Layer transition', fontsize=11)
ax2.set_ylabel('$\\|B^{(l)}\\|_F$', fontsize=11)
ax2.set_title('Grade-2: Bivector Norm (Rotation)', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

fig.suptitle(f'Grade Decomposition — {model.name}', fontsize=13, y=1.02)
fig.tight_layout()

save_path = os.path.join(os.getcwd(), 'ch06_grade_profile.png')
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()

# Print summary statistics
print(f"\n── Grade Comparison Summary ──")
print(f"  Mean ||P-I||_F (grade-0): {grade0_norms.mean():.4f}")
print(f"  Mean ||B||_F   (grade-2): {grade2_norms.mean():.4f}")
print(f"  Ratio grade-0/grade-2:    {grade0_norms.mean() / (grade2_norms.mean() + 1e-12):.2f}")
print(f"  → Grade-0 (stretching) is typically larger than grade-2 (rotation).")

## The Three-Phase Structure and Directional Flow Ratio

When you look at all three GA signatures together (rotation angle, condition number,
effective rank), a consistent pattern emerges — the **three-phase structure**:

1. **Early layers (embedding):** Large rotations, low κ, high erank → grade-2 (rotation) dominates
2. **Middle layers (interaction):** Moderate rotation, rising κ, curvature peaks → grade-0 and grade-2 both contribute
3. **Late layers (readout):** Small rotation, large κ, low erank → grade-0 (stretching) dominates

The **directional flow ratio** $\mathcal{R} = \|A\|_F / \|S\|_F$ quantifies this:
it's the grade-2 to grade-0 ratio at each layer.

In [ ]:
# ── Directional flow ratio across layers ──────────────────────────
from layer_time_ga.algebra import directional_flow_ratio
import layer_time_geometry as ltg

rf = result1.rotor_field
H_w = result1.H_whitened
n_transitions = len(rf.decompositions)

flow_ratios = []
for i in range(n_transitions):
    l = rf.decompositions[i].layer_index
    op = ltg.layer_operator(H_w, l)
    R_l = directional_flow_ratio(op.T_op)
    flow_ratios.append(R_l)

# Plot with three-phase annotation
fig, ax = plt.subplots(figsize=(10, 4))
layers = [vd.layer_index for vd in rf.decompositions]
ax.plot(layers, flow_ratios, 'o-', color='#2196F3', markersize=4)
ax.set_xlabel('Layer')
ax.set_ylabel(r'$\mathcal{R} = \|A\|_F / \|S\|_F$')
ax.set_title('Directional Flow Ratio: Grade-2 / Grade-0 Balance')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='$\\mathcal{R}=1$ (equal)')

# Mark three phases
n = len(layers)
ax.axvspan(layers[0], layers[n//4], alpha=0.1, color='blue', label='Early (embedding)')
ax.axvspan(layers[n//4], layers[3*n//4], alpha=0.1, color='green', label='Middle (interaction)')
ax.axvspan(layers[3*n//4], layers[-1], alpha=0.1, color='red', label='Late (readout)')
ax.legend(fontsize=8)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"\nMean flow ratio: {np.mean(flow_ratios):.4f}")
print(f"Early layers (0-6):  {np.mean(flow_ratios[:7]):.4f}")
print(f"Middle layers (7-20): {np.mean(flow_ratios[7:21]):.4f}")
print(f"Late layers (21-26): {np.mean(flow_ratios[21:]):.4f}")
print("\n→ Grade-0 (stretching) dominates everywhere, but the ratio drops in late layers.")

## The Control Finding

Empirical ablation studies reveal an asymmetry between the two grades:

| Intervention | Effect on output |
|:---|:---|
| Modify **grade-0** (replace eigenvalues of $P$ with uniform values) | **~48%** change in output logits |
| Modify **grade-2** (replace $U$ with identity, remove rotation) | **~9%** change in output logits |

**Why this asymmetry?** The explanation is geometric:

1. **Rotation preserves norms.** When the rotor acts on a hidden state, $\|R\,v\,R^{-1}\| = \|v\|$. The total "energy" of the representation is unchanged. Information is *redirected* but not *created or destroyed*.

2. **Stretching changes norms selectively.** The metric factor amplifies some directions and suppresses others. This is how the transformer *selects* which features matter and which to discard. It is a **lossy, irreversible** operation (in the information-theoretic sense).

3. **Condition number as selectivity.** A high condition number $\kappa$ means the layer is being very selective — strongly favoring certain directions. This selectivity is what drives the output change.

The condition number $\kappa(l)$ and effective rank $\text{erank}(l)$ together characterize the "selectivity fingerprint" of each layer.

In [ ]:
# ── Condition number and effective rank across layers ──────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Condition number
ax1.plot(layer_indices, result1.condition_numbers, color='#AA3377',
         linewidth=2, marker='o', markersize=3)
ax1.fill_between(layer_indices, result1.condition_numbers,
                 alpha=0.15, color='#AA3377')
ax1.set_xlabel('Layer transition', fontsize=11)
ax1.set_ylabel('Condition number κ', fontsize=11)
ax1.set_title('Metric Selectivity κ(l)', fontsize=12)
ax1.grid(True, alpha=0.3)

# Mark the peak
peak_kappa = result1.condition_numbers.argmax()
ax1.axvline(layer_indices[peak_kappa], color='grey', linestyle='--',
            alpha=0.5, label=f'Peak at layer {layer_indices[peak_kappa]}')
ax1.legend(fontsize=9)

# Panel 2: Effective rank
ax2.plot(layer_indices, result1.effective_ranks, color='#66CCEE',
         linewidth=2, marker='s', markersize=3)
ax2.fill_between(layer_indices, result1.effective_ranks,
                 alpha=0.15, color='#66CCEE')
ax2.set_xlabel('Layer transition', fontsize=11)
ax2.set_ylabel('Effective rank', fontsize=11)
ax2.set_title('Metric Dimensionality erank(l)', fontsize=12)
ax2.grid(True, alpha=0.3)

# Mark the minimum (most selective layer)
min_erank = result1.effective_ranks.argmin()
ax2.axvline(layer_indices[min_erank], color='grey', linestyle='--',
            alpha=0.5, label=f'Min at layer {layer_indices[min_erank]}')
ax2.legend(fontsize=9)

fig.suptitle(f'Grade-0 Control Profile — {model.name}', fontsize=13, y=1.02)
fig.tight_layout()

save_path = os.path.join(os.getcwd(), 'ch06_control_profile.png')
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()

print(f"\n── Control Summary ──")
print(f"  Mean κ:     {result1.condition_numbers.mean():.2f}")
print(f"  Max κ:      {result1.condition_numbers.max():.2f} (layer {layer_indices[peak_kappa]})")
print(f"  Mean erank: {result1.effective_ranks.mean():.1f}")
print(f"  Min erank:  {result1.effective_ranks.min():.1f} (layer {layer_indices[min_erank]})")
print(f"  → Layers with high κ and low erank are the most selective.")

## Comparing Two Prompts

The versor decomposition is prompt-dependent — different inputs produce different rotor fields. By overlaying the angle profiles of two prompts, we can see how the transformer allocates its geometric resources differently depending on the task.

Questions to consider:
- Do both prompts produce the same overall shape of $\theta(l)$?
- Are the peak-angle layers the same?
- Where do the profiles diverge most — and does that correspond to layers where the model is doing task-specific processing?

In [ ]:
# ── Analyse a second prompt ────────────────────────────────────
prompt2 = "Translate the following English text to French: The cat sat on the mat"
result2 = ltg_ga.analyse(prompt2, model=model)

print(f"Prompt 1: \"{prompt1[:50]}...\"")
print(f"  Tokens: {result1.n_tokens}, Mean angle: {result1.angles.mean():.4f}")
print(f"Prompt 2: \"{prompt2[:50]}...\"")
print(f"  Tokens: {result2.n_tokens}, Mean angle: {result2.angles.mean():.4f}")

# ── Overlay rotor angle profiles ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

layers1 = np.arange(len(result1.angles))
layers2 = np.arange(len(result2.angles))

ax.plot(layers1, result1.angles, color='#4477AA', linewidth=2,
        marker='o', markersize=3, alpha=0.8,
        label=f'"{prompt1[:35]}..."')
ax.plot(layers2, result2.angles, color='#EE6677', linewidth=2,
        marker='s', markersize=3, alpha=0.8,
        label=f'"{prompt2[:35]}..."')

ax.fill_between(layers1, result1.angles, alpha=0.08, color='#4477AA')
ax.fill_between(layers2, result2.angles, alpha=0.08, color='#EE6677')

ax.set_xlabel('Layer transition', fontsize=11)
ax.set_ylabel('Rotation angle θ (rad)', fontsize=11)
ax.set_title(f'Rotor Angle Comparison — {model.name}', fontsize=12)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
fig.tight_layout()

save_path = os.path.join(os.getcwd(), 'ch06_compare_angles.png')
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"\nSaved: {save_path}")
plt.show()

# ── Where do the profiles differ most? ─────────────────────────
n_common = min(len(result1.angles), len(result2.angles))
angle_diff = np.abs(result1.angles[:n_common] - result2.angles[:n_common])
max_diff_layer = angle_diff.argmax()
print(f"\n── Profile Comparison ──")
print(f"  Max divergence:  layer {max_diff_layer}, "
      f"Δθ = {angle_diff[max_diff_layer]:.4f} rad")
print(f"  Mean divergence: {angle_diff.mean():.4f} rad")
print(f"  Correlation:     {np.corrcoef(result1.angles[:n_common], result2.angles[:n_common])[0,1]:.3f}")
print(f"  → High correlation means the overall shape is model-driven;")
print(f"    the differences reflect prompt-specific processing.")

## Exercises

**Exercise 6.1 — Grade dominance ratio.**
For each layer transition, compute the ratio $\|P^{(l)} - I\|_F \;/\; \|B^{(l)}\|_F$. Plot this ratio across layers. At which layers does stretching dominate most? Does the pattern change for different prompts?

**Exercise 6.2 — Selectivity vs. rotation.**
Make a scatter plot of $\kappa(l)$ (condition number) vs. $\theta(l)$ (rotation angle) across all layers. Is there a correlation? Do high-selectivity layers also tend to have large rotations, or is there a trade-off?

**Exercise 6.3 — Three-prompt comparison.**
Run the analysis on three prompts of different types:
1. A factual statement ("The capital of Japan is Tokyo")
2. A mathematical expression ("Calculate the integral of x^2 from 0 to 1")
3. A creative prompt ("Write a poem about the ocean at night")

Overlay all three angle profiles and all three condition number profiles. Which prompt type shows the most geometric variation across layers?

**Exercise 6.4 — Cumulative rotation.**
Compose the rotors from layer 0 through layer $l$ to get the "cumulative rotor" $R_{\text{cum}}(l) = R^{(l)} \cdots R^{(1)} R^{(0)}$. Plot the cumulative rotation angle as a function of $l$. Does the cumulative angle grow linearly, or does it saturate? What does this tell you about whether successive layers rotate in similar or orthogonal planes?

**Exercise 6.5 — Effective rank and information bottleneck.**
Plot the effective rank $\text{erank}(l)$ across layers. Identify the layer with the smallest effective rank — this is the "information bottleneck" of the network. How does the bottleneck position change across different prompts? Relate this to the concept of the representation bottleneck in deep learning.